<a href="https://colab.research.google.com/github/JSJeong-me/LiteLLM-OnDeive-App/blob/main/0617-HFapi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade notebook jupyterlab ipywidgets nbformat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 83.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 913.0/913.0 kB 54.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 85.3 MB/s eta 0:00:00
  Attempting uninstall: widgetsnbextension
    Found existing installation: widgetsnbextension 3.6.10
    Uninstalling widgetsnbextension-3.6.10:
      Successfully uninstalled widgetsnbextension-3.6.10
  Attempting uninstall: ipywidgets
    Found existing installation: ipywidgets 7.7.1
   

In [ ]:
!pip install transformers datasets huggingface_hub

ERROR: Operation cancelled by user
^C


In [1]:
from huggingface_hub import notebook_login
notebook_login()

In [2]:
from transformers import pipeline

# 감정 분석 모델 호출
classifier = pipeline("sentiment-analysis")

# 예제 문장 분석
result = classifier("I love Huggingface API!")
print(result)

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

[{'label': 'POSITIVE', 'score': 0.9978556036949158}]


In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# 1. 모델과 토크나이저 이름 설정 (예: 다국어 감정 분석 모델)
model_checkpoint = "nlptown/bert-base-multilingual-uncased-sentiment"

# 2. 토크나이저 및 모델 로드
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint)

# 3. 입력 텍스트 준비
text = "이 제품은 정말 훌륭합니다! 사용하기 편리하고 성능도 좋네요."
inputs = tokenizer(text, return_tensors="pt")

# 4. 모델 추론
with torch.no_grad():
    logits = model(**inputs).logits

# 5. 결과 해석 (가장 높은 확률의 레이어 선택)
predicted_class_id = logits.argmax().item()
print(f"입력 문장: {text}")
print(f"예측된 별점/등급 (0-4): {predicted_class_id}")

config.json:   0%|          | 0.00/953 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/39.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/872k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/669M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

입력 문장: 이 제품은 정말 훌륭합니다! 사용하기 편리하고 성능도 좋네요.
예측된 별점/등급 (0-4): 4


In [4]:
from transformers import AutoImageProcessor, AutoModelForImageClassification
from PIL import Image
import requests

# 1. 이미지 모델 및 프로세서 로드 (Vision Transformer)
model_name = "google/vit-base-patch16-224"
processor = AutoImageProcessor.from_pretrained(model_name)
model = AutoModelForImageClassification.from_pretrained(model_name)

# 2. 테스트용 이미지 다운로드 (고양이 이미지)
url = "http://images.cocodataset.org/val2017/000000039769.jpg"
image = Image.open(requests.get(url, stream=True).raw)

# 3. 이미지 전처리
inputs = processor(images=image, return_tensors="pt")

# 4. 모델 추론
import torch
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# 5. 결과 해석
predicted_class_idx = logits.argmax(-1).item()
print("Predicted class:", model.config.id2label[predicted_class_idx])

preprocessor_config.json:   0%|          | 0.00/160 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/69.7k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Predicted class: Egyptian cat


In [6]:
from transformers import AutoProcessor, AutoModelForAudioClassification
import torch
import numpy as np

# 1. 오디오 모델 및 프로세서 로드 (Audio Spectrogram Transformer)
model_name = "MIT/ast-finetuned-audioset-10-10-0.4593"
# AutoAudioProcessor 대신 AutoProcessor를 사용해야 합니다.
processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForAudioClassification.from_pretrained(model_name)

# 2. 샘플 오디오 데이터 생성 (또는 파일 로드)
sampling_rate = 16000
duration = 2  # 2초
audio_dummy = np.random.uniform(-1, 1, sampling_rate * duration)

# 3. 오디오 전처리
inputs = processor(audio_dummy, sampling_rate=sampling_rate, return_tensors="pt")

# 4. 모델 추론
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# 5. 결과 해석
predicted_class_ids = torch.argmax(logits, dim=-1).item()
print(f"Predicted audio class ID: {predicted_class_ids}")
print(f"Predicted label: {model.config.id2label[predicted_class_ids]}")

preprocessor_config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

Predicted audio class ID: 515
Predicted label: Static


In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 모델 및 토크나이저 로드
model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 입력 문장 준비
input_text = "Hello, Huggingface API is"
inputs = tokenizer(input_text, return_tensors="pt")

# 모델을 이용한 텍스트 생성
outputs = model.generate(**inputs, max_length=50)
generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generated_text)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Hello, Huggingface API is now available.

The API is now available. The API is now available in the API Reference.

The API Reference is now available in the API Reference.

The API Reference is now available in


In [ ]:
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
# Llama 3.2 모델 실행
!ollama run llama3:latest

# Gemma 1B 모델 실행
!ollama run gemma:1b

# Gemma 3B 모델 실행
!ollama run gemma:3b

In [ ]:
# Start the Ollama server
# !ollama serve &

In [ ]:
import requests

def generate_text(model, prompt):
    url = "http://localhost:11434/api/generate"
    data = {
        "model": model,
        "prompt": prompt,
        "stream": False
    }
    response = requests.post(url, json=data)
    return response.json()["response"]

# Llama 3.2 모델로 텍스트 생성 예시
llama_response = generate_text("llama3", "Explain the concept of generative AI.")
print("Llama 3.2 모델 응답:\n", llama_response)

# Gemma 1B 모델로 텍스트 생성 예시
gemma_1b_response = generate_text("gemma:1b", "What is machine learning?")
print("Gemma 1B 모델 응답:\n", gemma_1b_response)

# Gemma 3B 모델로 텍스트 생성 예시
gemma_3b_response = generate_text("gemma:3b", "Describe the importance of data science.")
print("Gemma 3B 모델 응답:\n", gemma_3b_response)